In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

from src.data.data_loader import get_dataset
from src.features.graph_metrics import compute_dataset_metrics
from src.visualization.graphs import plot_degree_rank
from src.visualization.distributions import plot_degree_distribution

In [ ]:
datasets_to_load = ['cora', 'pubmed', 'facebook', 'twitch_ru', 'lastfm_asia',
                    'ogbn_arxiv', 'reddit', 'youtube']

graphs = {}
for name in datasets_to_load:
    ds = get_dataset(name, verbose=True)
    graphs[name] = ds

In [ ]:
metrics = {}
for name in graphs:
    m = compute_dataset_metrics(name, force_recompute=False)
    metrics[name] = {k: v for k, v in m.items() if isinstance(v, (int, float, str))}

df_metrics = pd.DataFrame(metrics).T.round(4)
df_metrics.to_csv('results/baseline_metrics.csv')
df_metrics

In [ ]:
for name, ds in graphs.items():
    plot_degree_rank(ds['graph'], name=name, show=True, save=True)

In [ ]:
for name, ds in graphs.items():
    plot_degree_distribution(
        ds['graph'],
        title=f'Распределение степеней — {name.upper()}',
        save_path=f"figures/degree_dist_{name}.png"
    )

In [ ]:
degree_dict = {name: [d for _, d in ds['graph'].degree()] for name, ds in graphs.items()}

plt.figure(figsize=(12, 7))
for name, degs in degree_dict.items():
    sns.kdeplot(degs, label=name.upper(), linewidth=2)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Степень вершины (log)')
plt.ylabel('Плотность')
plt.title('Сравнение распределений степеней всех графов')
plt.legend()
plt.tight_layout()
plt.savefig('figures/degree_distribution_comparison.png', dpi=300, bbox_inches='tight')
plt.show()